In [ ]:
import torch, random, io, sys, warnings
import os, numpy as np, pandas as pd, pyreadr
from tqdm import tqdm
import sys, os
sys.path.append(os.path.abspath(".."))

from cpd_model import parse_args, learn_one_seq_penalty

warnings.filterwarnings("ignore")

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {device}")

# ======================================================
# CONFIG
# ======================================================
REPS = 10
TRUE_CP = [100, 200]
Z_LIST = [1, 20]
TOL = 10

# ======================================================
# PREPARE BASE ARGS (template)
# ======================================================
base_args = parse_args()
base_args.epoch = 150
base_args.K_dim = 2
base_args.decoder_lr = 0.01
base_args.decoder_iteration = 20
base_args.langevin_s = 0.2
base_args.langevin_K = 100
base_args.kappa = 0.8
base_args.penalties = [0.01, 0.05, 0.1, 1]
base_args.nu_iteration = 20
base_args.output_layer = [50, 50]
base_args.scale_delta = False
base_args.signif_level = 0.99
base_args.true_CP_full = TRUE_CP

# ======================================================
# MAIN LOOP
# ======================================================
records = []

GLOBAL_SEED = 1

for z_dim in Z_LIST:
    print(f"\n==============================")
    print(f" Running z_dim = {z_dim}")
    print(f"==============================")

    for rep in range(1, REPS + 1):

        print(f"\n--- Replicate {rep}/{REPS} ---")

        # ---------- seed ----------
        SEED = GLOBAL_SEED + rep
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)

        # ---------- load data ----------
        Y = pyreadr.read_r(f"../real_data_sim/sim_dat_ult_5_{rep}.RDS")
        X = pyreadr.read_r(f"../real_data_sim/sim_x_ult_5_{rep}.RDS")

        Y_df = np.array(list(Y.values())[0])
        X_df = np.array(list(X.values())[0])

        # expand X
        X_rep = np.repeat(X_df[:, np.newaxis, :], 100, axis=1)
        Y = Y_df[:, :, 0:3]
        X = X_rep

        # ---------- build args ----------
        args = parse_args()
        args.__dict__.update(base_args.__dict__)

        args.z_dim = z_dim
        args.x_dim = X.shape[2]
        args.y_dim = Y.shape[2]
        args.num_time = X.shape[0]
        args.num_samples = X.shape[1]

        # ---------- tensors ----------
        x_input = torch.tensor(X, dtype=torch.float32).to(device)
        y_input = torch.tensor(Y, dtype=torch.float32).to(device)

        # ---------- split ----------
        odd_idx = range(1, args.num_time, 2)
        even_idx = range(0, args.num_time, 2)

        x_train = x_input[odd_idx].reshape(-1, args.x_dim)
        x_test  = x_input[even_idx].reshape(-1, args.x_dim)
        y_train = y_input[odd_idx].reshape(-1, args.y_dim)
        y_test  = y_input[even_idx].reshape(-1, args.y_dim)

        # ======================================================
        # penalty selection
        # ======================================================
        results_half = []

        for penalty in args.penalties:
            _stdout = sys.stdout
            # sys.stdout = io.StringIO()
            try:
                loss, pen = learn_one_seq_penalty(
                    args, x_train, y_train, x_test, y_test,
                    penalty=penalty, half=True
                )
            finally:
                sys.stdout = _stdout

            results_half.append([loss, pen])

        results_half = np.array(results_half)
        best_idx = np.argmin(results_half[:, 0])
        best_penalty = args.penalties[best_idx]

        print(f"[INFO] Best penalty = {best_penalty}")

        # ======================================================
        # full training
        # ======================================================
        _stdout = sys.stdout
        sys.stdout = io.StringIO()
        try:
            out = learn_one_seq_penalty(
                args,
                x_input.reshape(-1, args.x_dim),
                y_input.reshape(-1, args.y_dim),
                x_input.reshape(-1, args.x_dim),
                y_input.reshape(-1, args.y_dim),
                penalty=best_penalty,
                half=False
            )
            result = out[0]
        finally:
            sys.stdout = _stdout

        torch.cuda.empty_cache()

        # ======================================================
        # evaluation
        # ======================================================
        est_cp = np.array(result[5], dtype=int) if len(result[5]) > 0 else np.array([])
        true_cp = np.array(TRUE_CP)

        if len(est_cp) == 0:
            cover_rate = 0
            avg_dist = np.nan
            FP = 0
            FN = len(true_cp)
        else:
            dist_mat = np.abs(est_cp[:, None] - true_cp[None, :])
            min_dist_true = dist_mat.min(axis=0)
            min_dist_est  = dist_mat.min(axis=1)

            cover_rate = np.mean(min_dist_true <= TOL)
            avg_dist   = np.mean(min_dist_true)
            FP = np.sum(min_dist_est > TOL)
            FN = np.sum(min_dist_true > TOL)

        # ======================================================
        # record
        # ======================================================
        records.append({
            "z_dim": z_dim,
            "rep": rep,
            "best_penalty": best_penalty,
            "num_detected": len(est_cp),

            # core output
            "est_CP": str(list(est_cp)),
            "true_CP": str(TRUE_CP),

            # optional (keep if you want)
            "CE": result[0],   # count error
        })


# ======================================================
# SAVE RESULTS
# ======================================================
df = pd.DataFrame(records)
zlist_str = "_".join(map(str, Z_LIST))
out_name = f"cpd_zdim_experiment_{zlist_str}.csv"

df.to_csv(out_name, index=False)


[INFO] Using device: cuda

 Running z_dim = 1

--- Replicate 1/10 ---
Epoch   5 | Loss=608.722961 | Kurtosis=4.555623
Epoch  10 | Loss=585.711548 | Kurtosis=3.976328
Epoch  15 | Loss=567.415161 | Kurtosis=13.728583
Epoch  20 | Loss=557.285645 | Kurtosis=21.113327
Epoch  25 | Loss=550.989441 | Kurtosis=14.564590
Epoch  30 | Loss=548.393433 | Kurtosis=17.313822
Epoch  35 | Loss=546.466675 | Kurtosis=18.570711
Epoch  40 | Loss=545.503601 | Kurtosis=19.148447
Epoch  45 | Loss=544.697998 | Kurtosis=26.550526
Epoch  50 | Loss=545.175903 | Kurtosis=26.020447
Epoch  55 | Loss=549.003357 | Kurtosis=28.072964
Epoch  60 | Loss=546.650940 | Kurtosis=27.855997
Epoch  65 | Loss=545.102722 | Kurtosis=26.023352
Epoch  70 | Loss=544.393372 | Kurtosis=29.883406
Epoch  75 | Loss=544.341736 | Kurtosis=25.842554
Epoch  80 | Loss=543.485901 | Kurtosis=26.540590
Epoch  85 | Loss=544.099182 | Kurtosis=30.152979
Epoch  90 | Loss=543.200195 | Kurtosis=31.330811
Epoch  95 | Loss=543.551697 | Kurtosis=28.523895
E

In [15]:
import numpy as np
import pandas as pd
import re

# =====================================================
# CONFIG
# =====================================================
CSV_PATH = "cpd_zdim_experiment_2510.csv"
TRUE_CP = [100, 200]
TOL = 10
T = 296


# =====================================================
# 1️⃣ Robust parser
# =====================================================
def parse_cp(x):
    if pd.isna(x):
        return []
    
    x = str(x)

    nums = re.findall(r"np\.int64\((\d+)\)", x)

    return list(map(int, nums))


# =====================================================
# 2️⃣ Evaluation
# =====================================================
def evaluate_cp(est_cp, true_cp, tol, T):
    est_cp = np.array(est_cp, dtype=int)
    true_cp = np.array(true_cp, dtype=int)
    if len(est_cp) == 0:
        FP = 0
        FN = len(true_cp)
        Dt_e = np.inf
        De_t = np.inf
        CE = abs(len(est_cp) - len(true_cp))
        CS = 0.0
        return FP, FN, Dt_e, De_t, CE, CS

    dist_mat = np.abs(est_cp[:, None] - true_cp[None, :])

    min_dist_true = dist_mat.min(axis=0)
    min_dist_est  = dist_mat.min(axis=1)

    # ---------------------------
    # FP / FN（coverage-based）
    # ---------------------------
    FP = np.sum(min_dist_est > tol)
    FN = np.sum(min_dist_true > tol)

    # ---------------------------
    # Hausdorff-like
    # ---------------------------
    Dt_e = np.max(min_dist_true)
    De_t = np.max(min_dist_est)

    # ---------------------------
    # Cardinality error
    # ---------------------------
    CE = abs(len(est_cp) - len(true_cp))

    # ---------------------------
    # Segmentation score (CS)
    # ---------------------------
    def get_segments(cp, T):
        cp = np.sort(cp)
        segs = []
        prev = 0
        for c in cp:
            segs.append(set(range(prev, c)))
            prev = c
        segs.append(set(range(prev, T)))
        return segs

    G  = get_segments(true_cp, T)
    Gp = get_segments(est_cp, T)

    def jaccard(A, B):
        return len(A & B) / len(A | B) if len(A | B) > 0 else 0

    CS = 0
    for A in G:
        best = max(jaccard(A, Ap) for Ap in Gp)
        CS += len(A) * best
    CS /= T

    return FP, FN, Dt_e, De_t, CE, CS


# =====================================================
# 3️⃣ Main pipeline
# =====================================================
df = pd.read_csv(CSV_PATH)

# parse est_CP
df["est_CP"] = df["est_CP"].apply(parse_cp)


df = df.drop(columns=["CE"], errors="ignore")


metrics = df["est_CP"].apply(
    lambda cp: evaluate_cp(cp, TRUE_CP, TOL, T)
)

metrics_df = pd.DataFrame(
    metrics.tolist(),
    columns=["FP", "FN", "Dt_e", "De_t", "CE", "CS"]
)

df = pd.concat([df, metrics_df], axis=1)

# =====================================================
# 4️⃣ Summary
# =====================================================
summary = (
    df.groupby("z_dim")[["FP","FN","Dt_e","De_t","CE","CS"]]
    .mean()
    .round(2)
    .reset_index()
)

print(summary)

FileNotFoundError: [Errno 2] No such file or directory: 'cpd_zdim_experiment_2510.csv'